In [1]:
from transformers import pipeline


In [2]:
!pip install -q transformers datasets torch accelerate gradio ffmpeg screen-recorder


  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.4/485.4 kB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.4/100.4 kB 10.0 MB/s eta 0:00:00


In [3]:
!free -h


               total        used        free      shared  buff/cache   available
Mem:            12Gi       1.4Gi       1.1Gi       3.0Mi        10Gi        10Gi
Swap:             0B          0B          0B


In [4]:
from huggingface_hub import login

login()

In [5]:
!huggingface-cli whoami

Gakaxy


In [6]:
import os
import torch
import gradio as gr
import ffmpeg
import datetime
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline


In [7]:
!pip install streamlit

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 95.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 67.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 9.5 MB/s eta 0:00:00


In [8]:
!pip install bitsandbytes
!pip install --upgrade transformers accelerate

In [9]:
!free -h


               total        used        free      shared  buff/cache   available
Mem:            12Gi       1.9Gi       403Mi       3.0Mi        10Gi        10Gi
Swap:             0B          0B          0B


In [10]:
import torch
print("CUDA Available:", torch.cuda.is_available())
print("GPU Name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU found")

CUDA Available: True
GPU Name: Tesla T4


In [11]:
!pip uninstall -y bitsandbytes
!pip install --no-cache-dir bitsandbytes
!pip install --upgrade transformers accelerate

Found existing installation: bitsandbytes 0.45.3
Uninstalling bitsandbytes-0.45.3:
  Successfully uninstalled bitsandbytes-0.45.3
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 142.8 MB/s eta 0:00:00


In [12]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

MODEL_NAME = "mistralai/Mistral-7B-v0.1"

quantization_config = BitsAndBytesConfig(load_in_4bit=True)  # Use 4-bit quantization

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=quantization_config
)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [13]:
!free -h


               total        used        free      shared  buff/cache   available
Mem:            12Gi       2.6Gi       250Mi        15Mi       9.8Gi       9.8Gi
Swap:             0B          0B          0B


In [14]:
from transformers import pipeline


In [15]:
# Create chat pipeline
chatbot = pipeline("text-generation", model=model, tokenizer=tokenizer, max_new_tokens=256)

# Function for chatbot response
def chat_with_ai(user_input, history=[]):
    full_prompt = f"User: {user_input}\nAI:"
    response = chatbot(full_prompt)[0]['generated_text'].split("AI:")[-1].strip()
    history.append((user_input, response))
    return history, response


Device set to use cuda:0


In [16]:
import torch

# Function for chatbot response using model.generate()
def chat_with_ai(user_input, history=[]):
    full_prompt = f"User: {user_input}\nAI:"

    # Tokenize input
    inputs = tokenizer(full_prompt, return_tensors="pt").to("cuda")

    # Generate response
    output = model.generate(**inputs, max_new_tokens=256)

    # Decode and format response
    response = tokenizer.decode(output[0], skip_special_tokens=True).split("AI:")[-1].strip()

    history.append((user_input, response))
    return history, response


In [17]:
!pip install --upgrade gradio

In [18]:
import torch
print("CUDA Available:", torch.cuda.is_available())
print("GPU Name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU found")


CUDA Available: True
GPU Name: Tesla T4


In [19]:
!free -h


               total        used        free      shared  buff/cache   available
Mem:            12Gi       2.1Gi       1.1Gi        15Mi       9.5Gi        10Gi
Swap:             0B          0B          0B


In [20]:
import gradio as gr

# Function for handling chatbot messages
def handle_message(user_input, chat_history):
    chat_history, response = chat_with_ai(user_input, chat_history)
    return chat_history, ""

# Set up Gradio UI
with gr.Blocks() as demo:
    gr.Markdown("## 🚀 Chat with Your AI Replica!")
    chatbot_ui = gr.Chatbot()
    user_input = gr.Textbox(label="Enter your message", placeholder="Type here...")
    send_button = gr.Button("Send")

    send_button.click(handle_message, inputs=[user_input, chatbot_ui], outputs=[chatbot_ui, user_input])

# Launch the Gradio UI
demo.launch(share=True)


/usr/local/lib/python3.11/dist-packages/gradio/components/chatbot.py:285: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  warnings.warn(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9dbcc94284e5901e71.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
